# Graph Factory (Colab)

Build the master rich graph artifact once, then reuse it for training/sweeps.

This notebook only does graph creation from `data/consolidated_ff_local/*`.

## 1) Runtime Setup

In Colab, use GPU runtime. H100/A100 preferred for build speed and headroom.

In [ ]:
import os
import subprocess
import sys
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
print("python:", sys.version)
print("python_executable:", sys.executable)
if sys.version_info < (3, 10):
    raise RuntimeError("Python >= 3.10 is required.")
try:
    import torch
except Exception as exc:
    raise RuntimeError("PyTorch is required in Colab runtime.") from exc
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
print("cuda_version:", torch.version.cuda)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("gpu:", gpu_name)
else:
    print("WARNING: CUDA unavailable. Build can still run but may be slower.")


## 2) Mount Drive + Open Repo

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
repo_candidates = [
    "/content/drive/MyDrive/Forward-Risk-Manager",
    "/content/drive/MyDrive/forward-risk-manager",
    "/content/Forward-Risk-Manager",
    str(Path.cwd()),
]
repo = None
for cand in repo_candidates:
    p = Path(cand)
    if (p / "scripts" / "build_graphs.py").exists() and (p / "configs").exists():
        repo = p
        break
if repo is None:
    raise FileNotFoundError("Could not locate repo. Update repo_candidates in this cell.")
os.chdir(repo)
print("cwd:", Path.cwd())
print(subprocess.check_output(["bash", "-lc", "git rev-parse --short HEAD || true"], text=True))

## 3) Install/Validate Dependencies

In [ ]:
import importlib.util
import re
import sys
from pathlib import Path

if str(Path('src').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('src').resolve()))

from frisk.notebook_runtime import run_command, shell_quote

PYTHON = shell_quote(sys.executable)


def run(cmd: str, check: bool = True) -> int:
    result = run_command(
        cmd,
        allow_fail=not check,
        log_dir=Path('runs/experiments/_graph_factory_logs'),
    )
    return int(result.returncode)


run(f"{PYTHON} -m pip install --upgrade pip setuptools wheel")
# Lightweight essentials for graph build path.
need = []
for mod, pkg in [('pandas', 'pandas'), ('numpy', 'numpy'), ('tqdm', 'tqdm'), ('joblib', 'joblib')]:
    if importlib.util.find_spec(mod) is None:
        need.append(pkg)
if need:
    run(f"{PYTHON} -m pip install " + ' '.join(shell_quote(x) for x in sorted(set(need))))
if importlib.util.find_spec('torch') is None:
    run(f"{PYTHON} -m pip install torch")
# torch_geometric required by src/frisk/graph_builder.py
if importlib.util.find_spec('torch_geometric') is None:
    rc = run(f"{PYTHON} -m pip install torch-geometric", check=False)
    if rc != 0 or importlib.util.find_spec('torch_geometric') is None:
        import torch
        m = re.match(r'(\d+\.\d+\.\d+)', str(torch.__version__))
        torch_ver = m.group(1) if m else str(torch.__version__).split('+')[0]
        cuda_ver = getattr(torch.version, 'cuda', None)
        accel = f"cu{str(cuda_ver).replace('.', '')}" if cuda_ver else 'cpu'
        pyg_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{accel}.html"
        print('torch_geometric fallback wheel index:', pyg_url)
        for pkg in ['pyg_lib', 'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv']:
            run(f"{PYTHON} -m pip install {pkg} -f {shell_quote(pyg_url)}", check=False)
        run(f"{PYTHON} -m pip install torch-geometric", check=False)
if importlib.util.find_spec('torch_geometric') is None:
    raise RuntimeError('torch_geometric install failed. Restart runtime and rerun this cell.')
run(f"{PYTHON} -m pip install -e . --no-build-isolation")
run(f"{PYTHON} scripts/build_graphs.py --help")
print('Dependency setup complete.')


## 4) Build Configuration

In [ ]:
from pathlib import Path
import json
try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib
# Primary config for richer consolidated DATA FF graph.
BUILD_CONFIG = "configs/master_graph_ff.toml"
# Recommended stable output artifact for reuse in training notebooks.
BUILD_OUT = "data/processed/graphs_master_ff_rich.pt"
MANIFEST_OUT = BUILD_OUT + ".manifest.json"
# Leakage-safe build lags.
BUILD_CORR_LAG_DAYS = 1
BUILD_FEATURE_LAG_DAYS = 1
BUILD_MEMBERSHIP_LAG_DAYS = 1
# Lower this (e.g., 150) if too many windows are skipped for min_nodes.
BUILD_MIN_NODES_OVERRIDE = 100
# Colab-safe memory defaults.
BUILD_WORKERS = 1
BUILD_PARALLEL_BACKEND = "serial"
BUILD_JOBLIB_N_JOBS = 1
NO_PROGRESS = False
# Keep rich SEC fundamentals by default; fallback only if build fails.
ALLOW_NO_SEC_FALLBACK = True
FORCE_REBUILD = False
cfg_path = Path(BUILD_CONFIG)
if not cfg_path.exists():
    raise FileNotFoundError(cfg_path)
with cfg_path.open("rb") as f:
    cfg = tomllib.load(f).get("build_graphs", {})
required = [cfg.get("prices", "")]
if str(cfg.get("membership_mode", "")).strip().lower() != "all":
    required.append(cfg.get("constituents", ""))
if str(cfg.get("sec_as_fundamentals", True)).lower() in {"1", "true", "yes"}:
    required.append(cfg.get("sec_companyfacts", ""))
missing = [p for p in required if p and not Path(p).exists()]
if missing:
    raise FileNotFoundError("Missing build inputs: " + ", ".join(missing))
print(json.dumps({
    "build_config": BUILD_CONFIG,
    "build_out": BUILD_OUT,
    "workers": BUILD_WORKERS,
    "parallel_backend": BUILD_PARALLEL_BACKEND,
    "joblib_n_jobs": BUILD_JOBLIB_N_JOBS,
    "allow_no_sec_fallback": ALLOW_NO_SEC_FALLBACK,
    "force_rebuild": FORCE_REBUILD,
    "min_nodes_override": BUILD_MIN_NODES_OVERRIDE,
}, indent=2))

## 5) Fingerprint Check (Skip If Unchanged)

In [ ]:
from pathlib import Path
import json
import sys

if str(Path('src').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('src').resolve()))

from frisk.notebook_runtime import build_source_fingerprint, load_json

current_fp = build_source_fingerprint(
    Path(BUILD_CONFIG),
    tracked_keys=['prices', 'macro', 'fundamentals', 'sec_companyfacts', 'sec_submissions', 'constituents'],
    extra_fields={
        'lags': {
            'corr': int(BUILD_CORR_LAG_DAYS),
            'feature': int(BUILD_FEATURE_LAG_DAYS),
            'membership': int(BUILD_MEMBERSHIP_LAG_DAYS),
        },
        'workers': int(BUILD_WORKERS),
        'parallel_backend': str(BUILD_PARALLEL_BACKEND),
        'joblib_n_jobs': int(BUILD_JOBLIB_N_JOBS),
    },
)
manifest_path = Path(MANIFEST_OUT)
graphs_path = Path(BUILD_OUT)
existing_manifest = load_json(manifest_path, default=None)
UNCHANGED = (
    (not FORCE_REBUILD)
    and graphs_path.exists()
    and existing_manifest is not None
    and existing_manifest.get('source_fingerprint') == current_fp
)
print('graphs_path_exists:', graphs_path.exists())
print('manifest_exists:', manifest_path.exists())
print('unchanged:', UNCHANGED)


## 6) Build Graph Artifact

In [ ]:
from pathlib import Path
import sys

if str(Path('src').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('src').resolve()))

from frisk.notebook_runtime import run_command, shell_quote

PYTHON = shell_quote(sys.executable)


def run_cmd(cmd: str, allow_fail: bool = False) -> bool:
    result = run_command(
        cmd,
        allow_fail=allow_fail,
        env_overrides={'PYTHONUNBUFFERED': '1'},
        log_dir=Path('runs/experiments/_graph_factory_logs'),
    )
    print(f"elapsed_s={result.elapsed_s:.1f}, returncode={result.returncode}")
    return result.ok


if UNCHANGED:
    print('Fingerprint unchanged and graph artifact exists. Skipping build.')
else:
    Path(BUILD_OUT).parent.mkdir(parents=True, exist_ok=True)
    base = (
        f"{PYTHON} -u scripts/build_graphs.py --config {shell_quote(BUILD_CONFIG)} "
        f"--out {shell_quote(BUILD_OUT)} "
        f"--corr-lag-days {int(BUILD_CORR_LAG_DAYS)} "
        f"--feature-lag-days {int(BUILD_FEATURE_LAG_DAYS)} "
        f"--membership-lag-days {int(BUILD_MEMBERSHIP_LAG_DAYS)} "
        f"--workers {int(BUILD_WORKERS)} --parallel-backend {shell_quote(BUILD_PARALLEL_BACKEND)} "
        f"--joblib-n-jobs {int(BUILD_JOBLIB_N_JOBS)}"
    )
    if BUILD_MIN_NODES_OVERRIDE is not None:
        base += f" --min-nodes {int(BUILD_MIN_NODES_OVERRIDE)}"
    if NO_PROGRESS:
        base += ' --no-progress'
    else:
        base += ' --progress'
    ok = run_cmd(base, allow_fail=ALLOW_NO_SEC_FALLBACK)
    if (not ok) and ALLOW_NO_SEC_FALLBACK:
        print('Primary rich build failed. Retrying without SEC fundamentals for stability.')
        retry_cmd = base + ' --no-sec-as-fundamentals --feature-mode window_plus_summary'
        ok = run_cmd(retry_cmd, allow_fail=False)
    if not Path(BUILD_OUT).exists():
        raise FileNotFoundError(f'Build completed but artifact missing: {BUILD_OUT}')
print('graph artifact:', BUILD_OUT)


## 7) Validate + Write Manifest

In [ ]:
from pathlib import Path
import datetime as dt
import json
import torch

# PyTorch >=2.6 defaults torch.load(..., weights_only=True), which cannot load
# full PyG Data objects without explicit allowlisting. This artifact is locally
# produced by this notebook, so full-object load is expected here.
try:
    payload = torch.load(BUILD_OUT, map_location="cpu", weights_only=False)
except TypeError:
    # Backward compatibility with older torch versions without weights_only arg.
    payload = torch.load(BUILD_OUT, map_location="cpu")

graphs = payload.get("graphs", [])
dates = payload.get("dates", [])
stats = payload.get("stats", {})
cfg = payload.get("config", {})

summary = {
    "artifact": BUILD_OUT,
    "num_graphs": len(graphs),
    "date_start": dates[0] if dates else None,
    "date_end": dates[-1] if dates else None,
    "stats": stats,
    "feature_mode": cfg.get("feature_mode"),
}
print(summary)

total_windows = int(stats.get("total_windows", 0) or 0)
built = int(stats.get("built", 0) or 0)
if total_windows > 0:
    skipped_min_nodes = int(stats.get("skipped_min_nodes", 0) or 0)
    skipped_cols = int(stats.get("skipped_no_cols", 0) or 0)
    print(
        {
            "build_rate": round(built / total_windows, 4),
            "skip_rate": round((total_windows - built) / total_windows, 4),
            "skipped_min_nodes_rate": round(skipped_min_nodes / total_windows, 4),
            "skipped_no_cols_rate": round(skipped_cols / total_windows, 4),
            "min_nodes": cfg.get("min_nodes"),
        }
    )

manifest = {
    "created_at_utc": dt.datetime.utcnow().isoformat() + "Z",
    "artifact": BUILD_OUT,
    "num_graphs": len(graphs),
    "date_start": dates[0] if dates else None,
    "date_end": dates[-1] if dates else None,
    "build_stats": stats,
    "graph_config": cfg,
    "source_fingerprint": current_fp,
}
Path(MANIFEST_OUT).write_text(json.dumps(manifest, indent=2))
print("manifest:", MANIFEST_OUT)


## 8) Next Step

In your training notebook (`notebooks/colab_setup.ipynb`), set `RUN_BUILD=False` and point `train.graphs` to `data/processed/graphs_master_ff_rich.pt`.